# # Phase 5 — Research Mode Polymer Pipeline
# Lowest-possible error, multi-modal hybrid + ensemble (no runtime limits).
# Resume-friendly: every expensive step caches artifacts.


In [ ]:
# %% Cell 0.2 — Environment Check & Versions
import os, sys, json, time, platform, shutil, random
from pathlib import Path

def show_env():
    print("Python:", sys.version)
    print("Platform:", platform.platform())
    try:
        import torch
        print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
        if torch.cuda.is_available():
            print("CUDA device:", torch.cuda.get_device_name(0))
    except Exception as e:
        print("PyTorch not available:", e)
    for name in ["rdkit", "mordred", "transformers", "lightgbm", "sklearn", "pandas", "pyarrow"]:
        try:
            mod = __import__(name)
            print(f"{name}:", getattr(mod, "__version__", "OK"))
        except Exception as e:
            print(f"{name}: not importable ({e})")

show_env()


In [ ]:
# %% Cell 0.3 — Imports, Global Config & Paths
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

SEED = 42
TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"           # put train/test here
ART_DIR  = BASE_DIR / "artifacts"
for p in [ART_DIR, ART_DIR/"ingest", ART_DIR/"features", ART_DIR/"graphs", ART_DIR/"conformers",
          ART_DIR/"se3", ART_DIR/"lm", ART_DIR/"oof", ART_DIR/"meta", ART_DIR/"reports",
          ART_DIR/"models_lgb", ART_DIR/"models_gnn", ART_DIR/"models_lm", ART_DIR/"models_se3"]:
    p.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except: pass

set_seed()


In [ ]:
# %% Cell 0.4 — Utility Toolkit (Timer, IO, Column Fix, Logging)
import contextlib, time, json

class Timer(contextlib.ContextDecorator):
    def __init__(self, msg="⏱"):
        self.msg = msg
    def __enter__(self):
        self.t0 = time.time()
        print(self.msg, "...")
        return self
    def __exit__(self, *exc):
        dt = time.time() - self.t0
        print(f"{self.msg} done in {dt:.1f}s")

def save_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f: json.dump(obj, f, indent=2)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path) as f: return json.load(f)

def uniqueify_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Make columns unique by appending suffixes; preserves order."""
    seen, new_cols = {}, []
    for c in df.columns:
        if c not in seen:
            seen[c]=0; new_cols.append(c)
        else:
            seen[c]+=1; new_cols.append(f"{c}__dup{seen[c]}")
    df = df.copy()
    df.columns = new_cols
    return df

def safe_to_parquet(df: pd.DataFrame, path: Path):
    df2 = uniqueify_columns(df)
    df2.to_parquet(path)

def safe_read_csv(path):
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1")


In [ ]:
# %% Cell 1.1 — Load Core Data
TRAIN_PQ = ART_DIR/"ingest/train_raw.parquet"
TEST_PQ  = ART_DIR/"ingest/test_raw.parquet"
SUB_CSV  = ART_DIR/"ingest/sample_submission.csv"

if not TRAIN_PQ.exists():
    train_raw = safe_read_csv("competition/train.csv")
    test_raw  = safe_read_csv("competition/test.csv")
    sub       = safe_read_csv("competition/sample_submission.csv") if (DATA_DIR/"competition/sample_submission.csv").exists() else pd.DataFrame()
    train_raw.to_parquet(TRAIN_PQ)
    test_raw.to_parquet(TEST_PQ)
    if not sub.empty: sub.to_csv(SUB_CSV, index=False)
else:
    train_raw = pd.read_parquet(TRAIN_PQ)
    test_raw  = pd.read_parquet(TEST_PQ)
print(train_raw.shape, test_raw.shape, list(train_raw.columns))


In [ ]:
# %% Cell 1.3 — SMILES Standardization & Dedup
from rdkit import Chem

def standardize_smiles(s):
    if not isinstance(s, str): return None
    s = s.replace("*", "C")
    try:
        mol = Chem.MolFromSmiles(s)
        if mol is None: return None
        return Chem.MolToSmiles(mol, canonical=True)
    except:
        return None

TRAIN_STD_PQ = ART_DIR/"ingest/train.parquet"
TEST_STD_PQ  = ART_DIR/"ingest/test.parquet"

if not TRAIN_STD_PQ.exists():
    train = train_raw.copy()
    test  = test_raw.copy()
    train["SMILES_std"] = train["SMILES"].progress_apply(standardize_smiles)
    test["SMILES_std"]  = test["SMILES"].progress_apply(standardize_smiles)
    train = train.dropna(subset=["SMILES_std"]).drop_duplicates(subset=["SMILES_std"])
    test  = test.dropna(subset=["SMILES_std"]).drop_duplicates(subset=["SMILES_std"])
    train.to_parquet(TRAIN_STD_PQ)
    test.to_parquet(TEST_STD_PQ)
else:
    train = pd.read_parquet(TRAIN_STD_PQ)
    test  = pd.read_parquet(TEST_STD_PQ)

train.head(2)


In [ ]:
# %% Cell 1.4 — Target QC & Basic Cleaning
TRAIN_CLEAN_PQ = ART_DIR/"ingest/train_clean.parquet"

if not TRAIN_CLEAN_PQ.exists():
    df = train.copy()
    # basic sanity filters (keep flexible; adjust as needed)
    for t in TARGETS:
        if t in df.columns:
            # keep NaNs (we mask during training); just remove clearly invalid where present
            if t == "Density":
                bad = (df[t] <= 0)
                df.loc[bad, t] = np.nan
            if t == "Tg":
                bad = (df[t] <= 0)
                df.loc[bad, t] = np.nan
    df.to_parquet(TRAIN_CLEAN_PQ)
else:
    df = pd.read_parquet(TRAIN_CLEAN_PQ)
train = df


In [ ]:
# %% Cell 1.5 — Scaffold Grouping & Folds
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict

def murcko_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        core = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(core)
    except:
        return None

FOLDS_JSON = ART_DIR/"ingest/folds.json"

if not FOLDS_JSON.exists():
    scaff = train["SMILES_std"].progress_apply(murcko_scaffold).fillna("NONE")
    train["scaffold"] = scaff
    # GroupKFold by scaffold hash
    scaff2idx = defaultdict(list)
    for i, s in enumerate(scaff):
        scaff2idx[s].append(int(i))
    # create 5 folds rotating scaffold groups
    groups = list(scaff2idx.values())
    folds = [[] for _ in range(5)]
    for i, g in enumerate(groups):
        folds[i % 5].extend(g)
    folds = [sorted(v) for v in folds]
    save_json(FOLDS_JSON, {"folds": folds})
else:
    folds = load_json(FOLDS_JSON)["folds"]
len(folds), [len(f) for f in folds]


In [ ]:
# %% Cell 2.1 — Mordred Descriptors (optional but helpful)
from mordred import Calculator, descriptors
import warnings 
warnings.filterwarnings("ignore")
TRAIN_MORDRED = ART_DIR/"features/train_mordred.parquet"
TEST_MORDRED  = ART_DIR/"features/test_mordred.parquet"

if not TRAIN_MORDRED.exists():
    calc = Calculator(descriptors, ignore_3D=True)
    mols_tr = [Chem.MolFromSmiles(s) for s in train["SMILES_std"]]
    mols_te = [Chem.MolFromSmiles(s) for s in test["SMILES_std"]]
    with Timer("Computing Mordred (train)"):
        mord_tr = calc.pandas(mols_tr).apply(pd.to_numeric, errors="coerce")
    with Timer("Computing Mordred (test)"):
        mord_te = calc.pandas(mols_te).apply(pd.to_numeric, errors="coerce")
    # drop all-NaN and constant cols based on train
    keep = mord_tr.columns[(~mord_tr.isna().all()) & (mord_tr.nunique()>1)]
    mord_tr = mord_tr[keep]; mord_te = mord_te.reindex(columns=keep)
    # Safe write (fix duplicate names if any)
    safe_to_parquet(mord_tr.reset_index(drop=True), TRAIN_MORDRED)
    safe_to_parquet(mord_te.reset_index(drop=True), TEST_MORDRED)
else:
    mord_tr = pd.read_parquet(TRAIN_MORDRED)
    mord_te = pd.read_parquet(TEST_MORDRED)
mord_tr.shape, mord_te.shape


In [ ]:
# %% Cell 2.2 — RDKit Props & Morgan FPs
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
import warnings 
warnings.filterwarnings("ignore")
def rdkit_props(mol):
    # small but useful set; extend as needed
    return dict(
        MolWt=Descriptors.MolWt(mol),
        TPSA=Descriptors.TPSA(mol),
        NumHDonors=rdMolDescriptors.CalcNumHBD(mol),
        NumHAcceptors=rdMolDescriptors.CalcNumHBA(mol),
        NumAromaticRings=rdMolDescriptors.CalcNumAromaticRings(mol),
        FractionCSP3=rdMolDescriptors.CalcFractionCSP3(mol),
        HeavyAtomCount=mol.GetNumHeavyAtoms(),
        NumAliphaticRings=rdMolDescriptors.CalcNumAliphaticRings(mol),
        BertzCT=Descriptors.BertzCT(mol),
    )

def morgan_fp(mol, nBits=2048, radius=2):
    if mol is None: return np.zeros(nBits, dtype=np.uint8)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)
    arr = np.zeros((nBits,), dtype=np.uint8)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

TRAIN_RD = ART_DIR/"features/train_rd.parquet"
TEST_RD  = ART_DIR/"features/test_rd.parquet"

if not TRAIN_RD.exists():
    mols_tr = [Chem.MolFromSmiles(s) for s in train["SMILES_std"]]
    mols_te = [Chem.MolFromSmiles(s) for s in test["SMILES_std"]]
    with Timer("Computing RDKit props & Morgan (train)"):
        rows = []
        for m in tqdm(mols_tr, total=len(mols_tr)):
            props = rdkit_props(m)
            fp = morgan_fp(m)
            for i, b in enumerate(fp):
                props[f"morgan_{i}"] = int(b)
            rows.append(props)
        rd_tr = pd.DataFrame(rows)

    with Timer("Computing RDKit props & Morgan (test)"):
        rows = []
        for m in tqdm(mols_te, total=len(mols_te)):
            props = rdkit_props(m)
            fp = morgan_fp(m)
            for i, b in enumerate(fp):
                props[f"morgan_{i}"] = int(b)
            rows.append(props)
        rd_te = pd.DataFrame(rows)

    safe_to_parquet(uniqueify_columns(rd_tr), TRAIN_RD)
    safe_to_parquet(uniqueify_columns(rd_te), TEST_RD)
else:
    rd_tr = pd.read_parquet(TRAIN_RD)
    rd_te = pd.read_parquet(TEST_RD)
rd_tr.shape, rd_te.shape


In [ ]:
# %% Cell 2.3 — Assemble Tabular Matrices (+ impute + standardize)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

TRAIN_TAB = ART_DIR/"features/train_tabular.parquet"
TEST_TAB  = ART_DIR/"features/test_tabular.parquet"
TAB_META  = ART_DIR/"features/tabular_colmeta.json"

if not TRAIN_TAB.exists():
    tab_tr = pd.concat([mord_tr.reset_index(drop=True), rd_tr.reset_index(drop=True)], axis=1)
    tab_te = pd.concat([mord_te.reset_index(drop=True), rd_te.reset_index(drop=True)], axis=1)
    # align columns strictly
    tab_te = tab_te.reindex(columns=tab_tr.columns)
    # impute with train medians; standardize
    imputer = SimpleImputer(strategy="median")
    scaler  = StandardScaler(with_mean=True, with_std=True)
    Xtr = imputer.fit_transform(tab_tr)
    Xte = imputer.transform(tab_te)
    Xtr = scaler.fit_transform(Xtr)
    Xte = scaler.transform(Xte)
    tab_tr = pd.DataFrame(Xtr, columns=tab_tr.columns)
    tab_te = pd.DataFrame(Xte, columns=tab_tr.columns)
    # safe write
    safe_to_parquet(tab_tr, TRAIN_TAB)
    safe_to_parquet(tab_te, TEST_TAB)
    save_json(TAB_META, {"columns": list(tab_tr.columns)})
else:
    tab_tr = pd.read_parquet(TRAIN_TAB)
    tab_te = pd.read_parquet(TEST_TAB)
tab_tr.shape, tab_te.shape


In [ ]:
# %% Cell 3.2 — Approx 3D metrics (Rg proxy etc.) — optional
TRAIN_3D = ART_DIR/"features/train_3dfeats.parquet"
TEST_3D  = ART_DIR/"features/test_3dfeats.parquet"

if not TRAIN_3D.exists():
    # placeholder: simple proxies from 2D that correlate with size/flexibility
    # (true 3D conformers come later)
    def cheap3d_row(mol):
        return dict(
            nHeavy=mol.GetNumHeavyAtoms(),
            nBonds=mol.GetNumBonds(),
            ringCount=len(Chem.GetSymmSSSR(mol)),
        )
    mols_tr = [Chem.MolFromSmiles(s) for s in train["SMILES_std"]]
    mols_te = [Chem.MolFromSmiles(s) for s in test["SMILES_std"]]
    rows_tr = [cheap3d_row(m) for m in tqdm(mols_tr, total=len(mols_tr))]
    rows_te = [cheap3d_row(m) for m in tqdm(mols_te, total=len(mols_te))]
    pd.DataFrame(rows_tr).to_parquet(TRAIN_3D)
    pd.DataFrame(rows_te).to_parquet(TEST_3D)
else:
    pass


In [ ]:
# %% Cell 4.1 — Tokenization stats (lengths) — cheap signals
TRAIN_TOK = ART_DIR/"features/train_tokstats.parquet"
TEST_TOK  = ART_DIR/"features/test_tokstats.parquet"

if not TRAIN_TOK.exists():
    def tokstats(s):
        return dict(smiles_len=len(s), tokens=len(list(s)))
    ts_tr = train["SMILES_std"].progress_apply(tokstats).apply(pd.Series)
    ts_te = test["SMILES_std"].progress_apply(tokstats).apply(pd.Series)
    ts_tr.to_parquet(TRAIN_TOK)
    ts_te.to_parquet(TEST_TOK)
else:
    ts_tr = pd.read_parquet(TRAIN_TOK)
    ts_te = pd.read_parquet(TEST_TOK)


In [ ]:
# %% Cell 4.2 — (Optional) Frozen LM embeddings (ChemBERTa) — can be skipped initially
# If you want to run later, just toggle RUN_LM to True.
RUN_LM = False
TRAIN_LM = ART_DIR/"lm/train_lm_emb.npy"
TEST_LM  = ART_DIR/"lm/test_lm_emb.npy"

if RUN_LM and (not Path(TRAIN_LM).exists()):
    from transformers import AutoTokenizer, AutoModel
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "seyonec/ChemBERTa-zinc-base-v1"  # working public model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()
    def embed_batch(smiles):
        with torch.no_grad():
            tok = tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
            out = model(**tok).last_hidden_state.mean(dim=1)
            return out.detach().cpu().numpy()
    emb_tr = []
    bs = 64
    s_list = train["SMILES_std"].tolist()
    for i in tqdm(range(0, len(s_list), bs)):
        emb_tr.append(embed_batch(s_list[i:i+bs]))
    emb_tr = np.vstack(emb_tr)
    np.save(TRAIN_LM, emb_tr)
    # test
    emb_te = []
    s_list = test["SMILES_std"].tolist()
    for i in tqdm(range(0, len(s_list), bs)):
        emb_te.append(embed_batch(s_list[i:i+bs]))
    emb_te = np.vstack(emb_te)
    np.save(TEST_LM, emb_te)


In [ ]:
# %% Cell 7.1 — Metrics
def mae_per_target(y_true_df: pd.DataFrame, y_pred_df: pd.DataFrame):
    maes = {}
    for t in TARGETS:
        if t not in y_true_df.columns or t not in y_pred_df.columns:
            maes[t] = np.nan; continue
        m = y_true_df[t].notna() & y_pred_df[t].notna()
        if m.sum() == 0: maes[t] = np.nan; continue
        maes[t] = float(np.mean(np.abs(y_true_df.loc[m, t].values - y_pred_df.loc[m, t].values)))
    return maes

def iqr_scaled_wmae(y_true_df: pd.DataFrame, y_pred_df: pd.DataFrame):
    maes = mae_per_target(y_true_df, y_pred_df)
    norm = []
    for t in TARGETS:
        if np.isnan(maes.get(t, np.nan)): continue
        y = y_true_df[t].dropna().values
        q1, q3 = np.percentile(y, [25, 75])
        iqr = max(1e-8, q3-q1)
        norm.append(maes[t]/iqr)
    return float(np.mean(norm)) if norm else np.nan, maes


In [ ]:
# %% Cell 7.2 — Calibration helpers (linear + clipping)
from sklearn.linear_model import LinearRegression

def linear_calibrate(oof: np.ndarray, y_df: pd.DataFrame):
    """oof: [N, T] in TARGETS order. Returns calibrated oof, params per target."""
    o2 = oof.copy()
    params = {}
    for ti, t in enumerate(TARGETS):
        y = y_df[t].values if t in y_df else None
        p = oof[:, ti]
        mask = (~np.isnan(p)) & (~np.isnan(y))
        if mask.sum() < 20:
            params[t] = {"strategy":"skip"}; continue
        lr = LinearRegression().fit(p[mask].reshape(-1,1), y[mask])
        a, b = float(lr.coef_[0]), float(lr.intercept_)
        o2[mask, ti] = a*p[mask] + b
        params[t] = {"a":a,"b":b}
    return o2, params

def clip_by_quantiles(pred: np.ndarray, y_df: pd.DataFrame, q_low=0.5, q_hi=99.5):
    pred2 = pred.copy()
    limits = {}
    for ti, t in enumerate(TARGETS):
        if t not in y_df.columns: continue
        y = y_df[t].dropna().values
        lo, hi = np.percentile(y, [q_low, q_hi])
        pred2[:, ti] = np.clip(pred2[:, ti], lo, hi)
        limits[t] = (float(lo), float(hi))
    return pred2, limits


In [ ]:
# %% Cell 8.1 — Assemble Tabular Matrix for LGB
Y_TRAIN_PQ = ART_DIR/"features/y_train.parquet"
X_TR_NPY   = ART_DIR/"features/X_train_tab.npy"
X_TE_NPY   = ART_DIR/"features/X_test_tab.npy"

if not Path(X_TR_NPY).exists():
    X_tr = pd.concat([tab_tr.reset_index(drop=True),
                      pd.read_parquet(TRAIN_TOK)], axis=1)
    X_te = pd.concat([tab_te.reset_index(drop=True),
                      pd.read_parquet(TEST_TOK)], axis=1)
    # (Optionally add cheap 3D feats)
    if (ART_DIR/"features/train_3dfeats.parquet").exists():
        X_tr = pd.concat([X_tr, pd.read_parquet(ART_DIR/"features/train_3dfeats.parquet")], axis=1)
        X_te = pd.concat([X_te, pd.read_parquet(ART_DIR/"features/test_3dfeats.parquet")], axis=1)
    # (Optionally add LM embeddings)
    if Path(ART_DIR/"lm/train_lm_emb.npy").exists():
        emb_tr = np.load(ART_DIR/"lm/train_lm_emb.npy")
        emb_te = np.load(ART_DIR/"lm/test_lm_emb.npy")
        emb_cols = [f"lm_{i}" for i in range(emb_tr.shape[1])]
        X_tr = pd.concat([X_tr, pd.DataFrame(emb_tr, columns=emb_cols)], axis=1)
        X_te = pd.concat([X_te, pd.DataFrame(emb_te, columns=emb_cols)], axis=1)
    # save arrays
    np.save(X_TR_NPY, X_tr.values.astype(np.float32))
    np.save(X_TE_NPY, X_te.values.astype(np.float32))
    train[TARGETS].to_parquet(Y_TRAIN_PQ)

X_tr = np.load(X_TR_NPY)
X_te = np.load(X_TE_NPY)
y_train = pd.read_parquet(Y_TRAIN_PQ)
X_tr.shape, X_te.shape, y_train.shape


In [ ]:
# inside the per-target / per-fold loop, replace the lgb.train(...) call with:
cb = [
    lgb.early_stopping(stopping_rounds=200, first_metric_only=True),
    lgb.log_evaluation(period=200),
]

model = lgb.train(
    params,
    dtr,
    num_boost_round=10000,
    valid_sets=[dtr, dva],
    valid_names=["train", "valid"],
    callbacks=cb,  # <-- v4 style
)

# When predicting, be robust to v3/v4 attribute names:
best_iter = getattr(model, "best_iteration", None)
if best_iter is None:
    best_iter = getattr(model, "best_iteration_", None)

oof[val_idx[va_mask], ti] = model.predict(
    X_tr[val_idx][va_mask],
    num_iteration=best_iter
)
test_fold_pred.append(
    model.predict(X_te, num_iteration=best_iter)
)


In [ ]:
# %% Cell 8.3 — Calibrate + Clip LGB preds (no retraining)
OOF_LGB_CAL = ART_DIR/"oof/oof_lgb_calibrated.npy"
TEST_LGB_CAL= ART_DIR/"oof/test_lgb_calibrated.npy"

oof_cal, cal_params = linear_calibrate(oof, y_train)
oof_cal, clip_limits = clip_by_quantiles(oof_cal, y_train)
test_cal, _ = clip_by_quantiles(test_pred.copy(), y_train)

np.save(OOF_LGB_CAL, oof_cal)
np.save(TEST_LGB_CAL, test_cal)
w0, m0 = iqr_scaled_wmae(y_train, pd.DataFrame(oof, columns=TARGETS))
w1, m1 = iqr_scaled_wmae(y_train, pd.DataFrame(oof_cal, columns=TARGETS))
print("Before:", round(w0,6), m0)
print("After :", round(w1,6), m1)
save_json(ART_DIR/"oof/lgb_calibration.json", {"linear":cal_params, "clip":clip_limits})


In [ ]:
# %% Cell 9.1 — Graph Construction (2D) for PyTorch Geometric
# Produces cached tensors for fast training (you can also construct on-the-fly).
GR_TR = ART_DIR/"graphs/train_graph.pt"
GR_TE = ART_DIR/"graphs/test_graph.pt"

if not GR_TR.exists():
    from torch_geometric.data import Data
    import torch

    def atom_features(atom):
        return np.array([
            atom.GetAtomicNum(),
            atom.GetTotalDegree(),
            atom.GetFormalCharge(),
            int(atom.GetIsAromatic()),
            atom.GetTotalValence(),
        ], dtype=np.int32)

    def bond_features(bond):
        bt = bond.GetBondType()
        return np.array([
            int(bt==Chem.BondType.SINGLE),
            int(bt==Chem.BondType.DOUBLE),
            int(bt==Chem.BondType.TRIPLE),
            int(bt==Chem.BondType.AROMATIC),
            int(bond.GetIsConjugated()),
        ], dtype=np.int32)

    def smiles_to_data(s):
        mol = Chem.MolFromSmiles(s)
        x = np.vstack([atom_features(a) for a in mol.GetAtoms()]).astype(np.float32)
        edges = []
        eattr = []
        for b in mol.GetBonds():
            i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
            bf = bond_features(b)
            edges.append([i,j]); edges.append([j,i])
            eattr.append(bf); eattr.append(bf)
        edge_index = np.array(edges).T if edges else np.zeros((2,0), dtype=np.int64)
        edge_attr  = np.vstack(eattr).astype(np.float32) if eattr else np.zeros((0,5), dtype=np.float32)
        return Data(x=torch.from_numpy(x),
                    edge_index=torch.from_numpy(edge_index).long(),
                    edge_attr=torch.from_numpy(edge_attr).float())

    import pickle
    with Timer("Building graphs (train)"):
        g_tr = [smiles_to_data(s) for s in tqdm(train["SMILES_std"], total=len(train))]
    with Timer("Building graphs (test)"):
        g_te = [smiles_to_data(s) for s in tqdm(test["SMILES_std"], total=len(test))]
    torch.save(g_tr, GR_TR)
    torch.save(g_te, GR_TE)
else:
    pass


In [ ]:
# Convert existing PyG .pt graph lists into NumPy-safe dict archives (one-time)
import numpy as np, torch
from pathlib import Path
from torch_geometric.data import Data

SAFE_GR_TR = ART_DIR/"graphs/train_graph_safe.npz"
SAFE_GR_TE = ART_DIR/"graphs/test_graph_safe.npz"

def data_to_np(d: Data):
    # minimal fields we used (extend if you add more later)
    out = {}
    out["x"] = d.x.cpu().numpy().astype(np.float32) if d.x is not None else None
    out["edge_index"] = d.edge_index.cpu().numpy().astype(np.int64) if d.edge_index is not None else None
    out["edge_attr"]  = d.edge_attr.cpu().numpy().astype(np.float32) if getattr(d, "edge_attr", None) is not None else None
    return out

def np_to_data(o):
    import torch
    from torch_geometric.data import Data
    x = torch.from_numpy(o["x"]) if o["x"] is not None else None
    edge_index = torch.from_numpy(o["edge_index"]) if o["edge_index"] is not None else None
    edge_attr  = torch.from_numpy(o["edge_attr"]) if o["edge_attr"] is not None else None
    return Data(x=x, edge_index=edge_index.long(), edge_attr=edge_attr)

if not SAFE_GR_TR.exists() or not SAFE_GR_TE.exists():
    # load the existing .pt (using the hotfix)
    g_tr = torch.load(GR_TR, weights_only=False)
    g_te = torch.load(GR_TE, weights_only=False)

    # pack into arrays of objects (np.savez handles lists via object dtype)
    tr_objs = np.array([data_to_np(d) for d in g_tr], dtype=object)
    te_objs = np.array([data_to_np(d) for d in g_te], dtype=object)

    np.savez_compressed(SAFE_GR_TR, objs=tr_objs)
    np.savez_compressed(SAFE_GR_TE, objs=te_objs)
    print("Wrote:", SAFE_GR_TR, SAFE_GR_TE)
else:
    print("Safe graph archives already exist.")


In [ ]:
# Replace the torch.load(...) pair with this version-agnostic loader:
import numpy as np, torch
from torch_geometric.data import Data

SAFE_GR_TR = ART_DIR/"graphs/train_graph_safe.npz"
SAFE_GR_TE = ART_DIR/"graphs/test_graph_safe.npz"

def np_to_data(o):
    import torch
    from torch_geometric.data import Data
    x = torch.from_numpy(o["x"]) if o["x"] is not None else None
    edge_index = torch.from_numpy(o["edge_index"]) if o["edge_index"] is not None else None
    edge_attr  = torch.from_numpy(o["edge_attr"]) if o["edge_attr"] is not None else None
    return Data(x=x, edge_index=edge_index.long(), edge_attr=edge_attr)

if Path(SAFE_GR_TR).exists():
    tr = np.load(SAFE_GR_TR, allow_pickle=True)["objs"]
    te = np.load(SAFE_GR_TE, allow_pickle=True)["objs"]
    g_tr = [np_to_data(o.item() if isinstance(o, np.ndarray) else o) for o in tr]
    g_te = [np_to_data(o.item() if isinstance(o, np.ndarray) else o) for o in te]
else:
    # fallback to direct .pt load (with hotfix)
    g_tr = torch.load(GR_TR, weights_only=False)
    g_te = torch.load(GR_TE, weights_only=False)


In [ ]:
# ==== Cell 9.3 (REVISED): GNN K-fold training (PyG) with CUDA init, fixes, tuning for RTX 4060 Ti ====
import os, math, time, random
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- CUDA init & confirmation ----
CUDA_OK = torch.cuda.is_available()
if CUDA_OK:
    dev_name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f"[CUDA] Available: True | Device: {dev_name} | Capability: {cap} | PyTorch: {torch.__version__}")
else:
    print(f"[CUDA] Available: False | Using CPU | PyTorch: {torch.__version__}")

device = "cuda" if CUDA_OK else "cpu"

# ---- PyG imports delayed (after torch/cuda init) ----
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from torch_geometric.nn import GINEConv, GINConv, global_mean_pool, BatchNorm

# ---------- Config (tuned for RTX 4060 Ti; adjust if OOM) ----------
SEED            = 42
N_FOLDS         = 5
EPOCHS          = 160                  # a bit longer; early stopping protects us
PATIENCE        = 24
LR              = 2.0e-3
WEIGHT_DECAY    = 1.0e-4
BATCH_SIZE      = 512 if CUDA_OK else 128   # 4060 Ti should handle 512 with this model; reduce to 384 if OOM
NUM_WORKERS     = 4
GRAD_CLIP_NORM  = 2.0
USE_AMP         = True                 # mixed precision for speed
FORCE_RETRAIN   = False                # set True to force retrain even if preds exist
MODEL_TAG       = "gnn_gine_v2_4060ti"

# ---------- Paths ----------
PREDS_DIR   = ART_DIR / "preds"
MODELS_DIR  = ART_DIR / "models_gnn"
PREDS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OOF_NPY     = PREDS_DIR / f"oof_{MODEL_TAG}.npy"
TEST_NPY    = PREDS_DIR / f"test_{MODEL_TAG}.npy"
TEST_MEAN   = PREDS_DIR / f"test_mean_{MODEL_TAG}.npy"
FOLD_CKPT   = lambda k: MODELS_DIR / f"{MODEL_TAG}_fold{k}.pt"

# ---------- Repro ----------
def set_seed(sd=SEED):
    random.seed(sd); np.random.seed(sd); torch.manual_seed(sd)
    if CUDA_OK:
        torch.cuda.manual_seed_all(sd)
set_seed(SEED)

# ---------- Robust graph loading (.npz preferred; .pt fallback) ----------
def load_graph_list():
    SAFE_GR_TR = ART_DIR/"graphs/train_graph_safe.npz"
    SAFE_GR_TE = ART_DIR/"graphs/test_graph_safe.npz"
    if SAFE_GR_TR.exists() and SAFE_GR_TE.exists():
        def np_to_data(o):
            if isinstance(o, np.ndarray) and o.dtype == object:
                o = o.item()
            x = torch.from_numpy(o["x"]) if o["x"] is not None else None
            edge_index = torch.from_numpy(o["edge_index"]) if o["edge_index"] is not None else None
            edge_attr  = torch.from_numpy(o["edge_attr"]) if o["edge_attr"] is not None else None
            return Data(x=x, edge_index=edge_index.long(), edge_attr=edge_attr)
        tr = np.load(SAFE_GR_TR, allow_pickle=True)["objs"]
        te = np.load(SAFE_GR_TE, allow_pickle=True)["objs"]
        g_tr = [np_to_data(o) for o in tr]
        g_te = [np_to_data(o) for o in te]
        return g_tr, g_te

    # Fallback: .pt with safe allowlisting and weights_only=False
    from torch.serialization import add_safe_globals
    try:
        from torch_geometric.data import Data as _Data
        from torch_geometric.data.data import DataEdgeAttr as _DataEdgeAttr
        add_safe_globals([_Data, _DataEdgeAttr])
    except Exception:
        pass
    g_tr = torch.load(GR_TR, weights_only=False)
    g_te = torch.load(GR_TE, weights_only=False)
    return g_tr, g_te

g_tr, g_te = load_graph_list()
assert len(g_tr) == len(train), f"train graphs ({len(g_tr)}) != train rows ({len(train)})"
assert len(g_te) == len(test),  f"test graphs ({len(g_te)}) != test rows ({len(test)})"

# ---------- Attach targets/masks (as [1, T] so PyG batches -> [B, T]) ----------
TARGETS = list(TARGETS)  # ensure list
Y_full  = torch.tensor(train[TARGETS].values, dtype=torch.float32)
M_full  = ~torch.isnan(Y_full)
Y_full[~M_full] = 0.0

for i, d in enumerate(g_tr):
    d.y = Y_full[i].unsqueeze(0)            # [1, T]
    d.m = M_full[i].to(torch.float32).unsqueeze(0)  # [1, T]

# ---------- NaN-robust target stats (no torch.nan* dependency) ----------
def _zero_like(x): return torch.zeros((), device=x.device, dtype=x.dtype)

def nanmean(x: torch.Tensor, dim=0, keepdim=False):
    mask = ~torch.isnan(x)
    num  = torch.where(mask, x, _zero_like(x)).sum(dim=dim, keepdim=keepdim)
    den  = mask.sum(dim=dim, keepdim=keepdim).clamp(min=1)
    return num / den

def nanstd(x: torch.Tensor, dim=0, keepdim=False, unbiased=False):
    m = nanmean(x, dim=dim, keepdim=True)
    mask = ~torch.isnan(x)
    diff2 = torch.where(mask, (x - m)**2, _zero_like(x)).sum(dim=dim, keepdim=keepdim)
    den   = mask.sum(dim=dim, keepdim=keepdim).clamp(min=1)
    if unbiased:
        den = torch.clamp(den - 1, min=1)
    var = diff2 / den
    return torch.sqrt(var)

y_mean = nanmean(Y_full, dim=0, keepdim=False)
y_std  = nanstd( Y_full, dim=0, keepdim=False).clamp_min(1e-6)

# (Optional) store z-scored targets in each graph for debugging
for d in g_tr:
    d.yz = (d.y - y_mean) / y_std

# ---------- Metrics ----------
@torch.no_grad()
def mae_per_target(y_true, y_pred, mask):
    diff = (y_true - y_pred).abs() * mask
    denom = mask.sum(dim=0).clamp_min(1.0)
    return diff.sum(dim=0) / denom

def iqr_normalized_wmae(y_true, y_pred, mask, iqr_dict):
    maes = mae_per_target(y_true, y_pred, mask)
    iqr = torch.tensor([iqr_dict[t] for t in TARGETS], device=maes.device, dtype=maes.dtype).clamp_min(1e-8)
    return (maes / iqr).mean().item(), {t: maes[i].item() for i, t in enumerate(TARGETS)}

# ---------- IQR via NumPy (robust across torch versions) ----------
if "iqr" not in globals():
    y_np = train[TARGETS].values.astype(np.float32)
    q75  = np.nanpercentile(y_np, 75, axis=0)
    q25  = np.nanpercentile(y_np, 25, axis=0)
    iqr  = {t: float(max(q75[i] - q25[i], 1e-8)) for i, t in enumerate(TARGETS)}

# ---------- Model (tuned) ----------
def mlp(d_in, d_hid, d_out):
    return nn.Sequential(nn.Linear(d_in, d_hid), nn.ReLU(inplace=True), nn.Linear(d_hid, d_out))

class GNN(nn.Module):
    def __init__(self, in_dim, edge_dim, hidden=384, out_dim=len(TARGETS), layers=4, dropout=0.10):
        super().__init__()
        self.edge_dim = edge_dim
        self.dropout  = dropout

        mlp1 = mlp(in_dim, hidden, hidden)
        if edge_dim is not None:
            self.conv1 = GINEConv(mlp1, train_eps=True, edge_dim=edge_dim)
        else:
            self.conv1 = GINConv(mlp1, train_eps=True)

        convs, bns = [], []
        for _ in range(layers - 1):
            block_mlp = mlp(hidden, hidden, hidden)
            if edge_dim is not None:
                conv = GINEConv(block_mlp, train_eps=True, edge_dim=edge_dim)
            else:
                conv = GINConv(block_mlp, train_eps=True)
            convs.append(conv); bns.append(BatchNorm(hidden))
        self.convs = nn.ModuleList(convs)
        self.bns   = nn.ModuleList(bns)

        self.head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        edge_attr = getattr(data, "edge_attr", None)
        x = self.conv1(x, edge_index, edge_attr) if self.edge_dim is not None else self.conv1(x, edge_index)
        x = F.relu(x)
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_attr) if self.edge_dim is not None else conv(x, edge_index)
            x = bn(x); x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        g = global_mean_pool(x, batch)
        return self.head(g)  # [B, T] (z-scored)

# Detect dims
sample = g_tr[0]
in_dim   = sample.x.size(-1) if sample.x is not None else 1
edge_dim = sample.edge_attr.size(-1) if getattr(sample, "edge_attr", None) is not None else None
print(f"[GNN] in_dim={in_dim}, edge_dim={edge_dim}, hidden=384, layers=4, dropout=0.10, BATCH={BATCH_SIZE}")

# ---------- DataLoaders ----------
def make_loader(graphs, idxs, shuffle=False):
    subset = [graphs[i] for i in idxs]
    return DataLoader(subset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=CUDA_OK)

# ---------- K-fold ----------
from sklearn.model_selection import KFold
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
indices = np.arange(len(g_tr))

# If predictions exist and not forcing retrain, just load and report
if OOF_NPY.exists() and TEST_MEAN.exists() and not FORCE_RETRAIN:
    oof_pred  = torch.tensor(np.load(OOF_NPY), dtype=torch.float32)
    test_pred = torch.tensor(np.load(TEST_MEAN), dtype=torch.float32)
    y_true    = torch.tensor(train[TARGETS].values, dtype=torch.float32)
    mask_true = (~torch.isnan(y_true)).to(torch.float32)
    iqr_w, _  = iqr_normalized_wmae(y_true, oof_pred, mask_true, {t: iqr[t] for t in TARGETS})
    print(f"[{MODEL_TAG}] (loaded) IQR‑normalized wMAE: {iqr_w:.6f}")
else:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    oof = torch.zeros((len(g_tr), len(TARGETS)), dtype=torch.float32)
    test_preds_folds = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(indices)):
        print(f"\n===== FOLD {fold+1}/{N_FOLDS} =====")
        tr_loader = make_loader(g_tr, tr_idx, shuffle=True)
        va_loader = make_loader(g_tr, va_idx, shuffle=False)
        te_loader = DataLoader(g_te, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=CUDA_OK)

        model = GNN(in_dim=in_dim, edge_dim=edge_dim, hidden=384, out_dim=len(TARGETS), layers=4, dropout=0.10).to(device)
        opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        sch   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR*0.1)

        class EarlyStopper:
            def __init__(self, patience=PATIENCE, mode="min", min_delta=1e-5):
                self.best = math.inf if mode=="min" else -math.inf
                self.patience = patience; self.mode = mode; self.min_delta = min_delta
                self.num_bad = 0; self.best_state = None
            def step(self, metric, model):
                improved = (metric < self.best - self.min_delta) if self.mode=="min" else (metric > self.best + self.min_delta)
                if improved:
                    self.best = metric; self.num_bad = 0
                    self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    return True
                self.num_bad += 1; return False
            def should_stop(self): return self.num_bad >= self.patience

        stopper = EarlyStopper()

        # Ensure stats on correct device
        y_mean_d = y_mean.to(device)
        y_std_d  = y_std.to(device)

        def run_epoch(loader, train_mode: bool):
            model.train(train_mode)
            total_loss, n_batches = 0.0, 0
            pbar = tqdm(loader, leave=False, desc="train" if train_mode else "valid")
            for batch in pbar:
                batch = batch.to(device, non_blocking=True)

                # Ensure labels/masks have shape [B, T]
                if batch.y.dim() == 1 and batch.y.numel() % len(TARGETS) == 0:
                    batch.y = batch.y.view(-1, len(TARGETS))
                if batch.m.dim() == 1 and batch.m.numel() % len(TARGETS) == 0:
                    batch.m = batch.m.view(-1, len(TARGETS))

                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    pred_z = model(batch)                          # [B, T] (z-scored)
                    yz  = (batch.y - y_mean_d) / y_std_d          # [B, T]
                    m   = batch.m                                  # [B, T]
                    loss = (pred_z - yz).abs() * m
                    denom = m.sum().clamp_min(1.0)
                    loss = loss.sum() / denom

                if train_mode:
                    opt.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    scaler.step(opt); scaler.update()

                total_loss += loss.item(); n_batches += 1
                if train_mode:
                    pbar.set_postfix(loss=f"{(total_loss/n_batches):.4f}", lr=f"{opt.param_groups[0]['lr']:.2e}")

            return total_loss / max(n_batches, 1)

        best_va = math.inf
        best_path = FOLD_CKPT(fold)
        last_improved_epoch = 0

        for epoch in range(1, EPOCHS+1):
            tl = run_epoch(tr_loader, True)
            vl = run_epoch(va_loader, False)
            sch.step()

            improved = stopper.step(vl, model)
            if improved:
                torch.save(stopper.best_state, best_path)
                best_va = vl; last_improved_epoch = epoch

            rem = max(0, EPOCHS - epoch)
            print(f"Epoch {epoch:03d} | train {tl:.4f} | valid {vl:.4f} | best {best_va:.4f} | ETA ~{rem} ep")
            if stopper.should_stop():
                print(f"Early stopping at epoch {epoch} (best @ {last_improved_epoch})")
                break

        # Load best and predict
        state = torch.load(best_path, map_location="cpu")
        model.load_state_dict(state); model.to(device); model.eval()

        with torch.no_grad():
            # Valid predictions -> OOF
            va_graphs = [g_tr[i] for i in va_idx]
            preds_va = []
            for batch in DataLoader(va_graphs, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=CUDA_OK):
                batch = batch.to(device, non_blocking=True)
                pz = model(batch)                                  # z-scored
                p  = pz * y_std_d + y_mean_d                       # de-standardize
                preds_va.append(p.detach().cpu())
            preds_va = torch.cat(preds_va, dim=0)
            oof[va_idx] = preds_va

            # Test predictions
            preds_te = []
            for batch in DataLoader(g_te, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=CUDA_OK):
                batch = batch.to(device, non_blocking=True)
                pz = model(batch)
                p  = pz * y_std_d + y_mean_d
                preds_te.append(p.detach().cpu())
            preds_te = torch.cat(preds_te, dim=0)  # [N_test, T]
            test_preds_folds.append(preds_te.numpy())

        print(f"Fold {fold} complete. ckpt -> {best_path}")

    # Save outputs
    np.save(OOF_NPY,  oof.numpy())
    np.save(TEST_NPY, np.stack(test_preds_folds, axis=0))                     # [K, N_test, T]
    np.save(TEST_MEAN, np.mean(np.stack(test_preds_folds, axis=0), axis=0))   # [N_test, T]

    # Metrics on OOF
    y_true    = torch.tensor(train[TARGETS].values, dtype=torch.float32)
    mask_true = (~torch.isnan(y_true)).to(torch.float32)
    iqr_w, maes = iqr_normalized_wmae(y_true, oof, mask_true, {t: iqr[t] for t in TARGETS})
    print(f"\n[{MODEL_TAG}] IQR‑normalized wMAE: {iqr_w:.6f}")
    print(f"[{MODEL_TAG}] Per‑target MAE (raw): " +
          str({t: mae_per_target(y_true, oof, mask_true)[i].item() for i, t in enumerate(TARGETS)}))
    print(f"OOF saved: {OOF_NPY}")
    print(f"Test folds (stack) saved: {TEST_NPY}")
    print(f"Test mean saved: {TEST_MEAN}")
